# EDA — AML Transaction Monitoring**Dataset:** IBM Transactions for Anti-Money Laundering (HI-Small), 5.08Mtransactions across 10 days, 0.089% laundering.**Purpose:** decide what goes into `features.py`. Every feature kept orrejected below has a measured reason.---### Two rules applied throughout**1. Features are computed on the FULL frame; target statistics on TRAIN only.**Account-history features look backwards — a day-8 transaction must see thataccount's day-1 activity, which sits in the training period. Computing themseparately per split would leave every test row with empty history.But anything that reads `is_laundering` (lift, decile rates, threshold sweeps)runs on training data alone. Otherwise test-set information influences featurechoices and inflates final metrics invisibly.**2. No feature may use information unavailable at scoring time.**The test: *could a bank compute this at 2pm on Tuesday using only what it knewbefore 2pm on Tuesday?* This rules out absolute dates, position-in-file, andany aggregate spanning the whole dataset.

---# 1. Setup

In [ ]:
%load_ext autoreload%autoreload 2import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.stats import chi2_contingency, fisher_exactfrom aml_detection.config import INTERIM_DATA_DIRfrom aml_detection.eda import (    overview, target_balance,    explore_categorical, explore_numeric, explore_boolean, explore_temporal,)pd.set_option("display.width", 160)pd.set_option("display.max_columns", 60)

In [ ]:
df = pd.read_parquet(INTERIM_DATA_DIR / "merged" / "transactions_merged.parquet")print(f"{len(df):,} rows x {df.shape[1]} columns")df.head(3)

---# 2. Data preparation findingsTwo structural decisions were made in `dataset_loader.py` before this notebook.Both came from inspecting the daily distribution, and both materially changewhat the model sees.

In [ ]:
per_day = df.groupby(df.timestamp.dt.date).agg(    rows=("is_laundering", "size"),    positives=("is_laundering", "sum"),    rate=("is_laundering", "mean"),)print(per_day.to_string())

### 2.1 Simulation tail removed (days 11-18)The raw file spans 1-18 September, but the two regions are not comparable:| period | rows/day | laundering rate ||---|---|---|| days 1-10 | ~500,000 | ~0.1% || days 11-18 | 400 down to 11 | **58-73%** |Background transaction generation stops after day 10. What remains is thetrailing legs of chains that started earlier — 1,108 rows, 0.02% of the file.**Why they were dropped rather than kept as extra positives:** a temporal splitwould place them in the test set, where a 58% positive rate produces metricsthat look excellent and mean nothing. Account features would also learn"active after 11 Sept ⇒ laundering", which is a calendar artifact.The chains themselves are not lost — their earlier legs remain in days 1-10.### 2.2 Burn-in period flagged (days 1-2)| period | rows | positives | rate ||---|---|---|---|| days 1-2 | 1.87M | 730 | **0.039%** || days 3-10 | 3.21M | 3,792 | **0.118%** |Three times less laundering in the opening two days. Chains take days tounfold, so early data is structurally short of laundering — the patterns havenot formed yet, not because those days are cleaner.**Decision:** flag, do not delete. Those rows are excluded as *trainingexamples* (mixing two base rates miscalibrates the model) but retained in theframe, because computing "transactions in the prior 7 days" for a day-3 rowrequires days 1-2 to look back at.

---# 3. Temporal splitA random split would leak: laundering is multi-step chains, so randomassignment puts earlier legs in train and later legs in test. A time-basedsplit mirrors deployment.

In [ ]:
SPLIT = df.timestamp.quantile(0.70)print(f"split at {SPLIT}")print(f"train  {(df.timestamp <= SPLIT).sum():,} rows")print(f"test   {(df.timestamp >  SPLIT).sum():,} rows")print("\n(actual split objects created in section 5, AFTER features are built)")

---# 4. Feature constructionBuilt on the full frame, before splitting.

### 4.1 Account history and network structure**The leakage rule:** every feature answers *what did the bank know about thisaccount immediately BEFORE this transaction?***On performance:** the obvious implementation(`groupby.apply(lambda s: s.shift().expanding().mean())`) runs a Pythonfunction per group — minutes across ~500K accounts. The version below is fullyvectorised: **5.4s on 3.5M rows**.**The expanding-nunique trick.** pandas has no "count distinct so far withingroup". Instead: mark the first appearance of each `(src, dst)` pair, thencumsum those markers per sender. A repeat pair contributes 0, so the runningcount only increases on genuinely new counterparties. Subtracting the currentrow's marker excludes the current transaction.Verified leakage-free by brute-force recomputation on random rows.

In [ ]:
def build_account_features(df: pd.DataFrame) -> pd.DataFrame:    """Account history features using ONLY prior transactions."""    df = df.sort_values("timestamp").reset_index(drop=True)    src = df.groupby("src_account", sort=False)    dst = df.groupby("dst_account", sort=False)    # ---- transaction counts so far ----    df["src_txn_count_prior"] = src.cumcount()    df["dst_txn_count_prior"] = dst.cumcount()    # ---- timing ----    df["src_secs_since_last"] = src.timestamp.diff().dt.total_seconds()    df["dst_secs_since_last"] = dst.timestamp.diff().dt.total_seconds()    df["src_secs_since_last_log"] = np.log1p(df.src_secs_since_last)    # ---- distinct counterparties so far (vectorised expanding nunique) ----    first_pair = ~df.duplicated(["src_account", "dst_account"])    running = first_pair.groupby(df.src_account, sort=False).cumsum()    df["src_unique_dst_prior"] = running - first_pair.astype(int)    first_pair_in = ~df.duplicated(["dst_account", "src_account"])    running_in = first_pair_in.groupby(df.dst_account, sort=False).cumsum()    df["dst_unique_src_prior"] = running_in - first_pair_in.astype(int)    # ---- fan ratios ----    # NaN below 3 prior transactions: 1/1 and 1/2 create spikes at 1.0 and 0.5    # that reflect newness, not fan-out behaviour    df["src_fanout_ratio"] = np.where(        df.src_txn_count_prior >= 3,        df.src_unique_dst_prior / df.src_txn_count_prior,        np.nan,    )    df["dst_fanin_ratio"] = np.where(        df.dst_txn_count_prior >= 3,        df.dst_unique_src_prior / df.dst_txn_count_prior,        np.nan,    )    return dfdf = build_account_features(df)print(df.shape)

### 4.2 Amount normalised within currencyRaw amount is not comparable across rows — currency medians span six orders ofmagnitude (Bitcoin 0.07, USD 962, Rupee 69,166, Yen 101,537) and no exchangerates are available.A plain z-score was rejected: with a maximum of 1e12, mean and standarddeviation are dominated by outliers — median z lands near −0.12 while themaximum reaches 122.

In [ ]:
df["amt_pct_ccy"] = (    df.groupby("currency_paid", observed=True).amount_paid.rank(pct=True))

### 4.3 Deviation from the account's own baseline*Is this transaction unusually large **for this account**?* A 50,000 transferis routine for a business and alarming for an account that has only ever sent500.**Three iterations were needed:**| attempt | formulation | result ||---|---|---|| 1 | `amount / prior arithmetic mean` | unusable, values to 3.5e7 || 2 | `log(amount / prior arithmetic mean)` | centred at **−3**, only 2.5× spread || 3 | `log(amount) − prior geometric mean of logs` | centred at 0, **3.5×** |**The −3 was the diagnostic.** A typical transaction appearing to be 5% of theaccount's average cannot be right. The arithmetic mean was being dragged upwardby occasional huge transactions, making every subsequent normal transactionlook small.Averaging in *log space* gives a geometric mean, which outliers cannotdominate. For an account sending 100, 100, 100, 1,000,000 — the arithmetic meanis 250,000, the geometric mean is ~560.**General lesson:** for multiplicative, heavy-tailed quantities like money,work in log space for both the value and the average.

In [ ]:
df["src_log_amount"] = np.log1p(df.amount_paid)g = df.groupby("src_account", sort=False)prior_logmean = (    (g.src_log_amount.cumsum() - df.src_log_amount)    / df.src_txn_count_prior.replace(0, np.nan))df["src_log_dev"] = np.where(    df.src_txn_count_prior >= 3,    df.src_log_amount - prior_logmean,    np.nan,)

### 4.4 TimeCyclical encoding so 23:00 and 00:00 are adjacent rather than 23 units apart.`is_hour_zero` is flagged separately: hour 0 carries 4× the volume of any otherhour, which almost certainly indicates a default timestamp for records withouta precise time rather than a genuine surge at midnight.

In [ ]:
df["hour"] = df.timestamp.dt.hourdf["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)df["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)df["is_hour_zero"] = df.hour == 0

---# 5. SplitFeatures are now built on the full frame. Everything below reads the target, soit runs on **train only**.

In [ ]:
train = df[df.timestamp <= SPLIT].copy()test  = df[df.timestamp >  SPLIT].copy()      # UNTOUCHED until final evaluationprint(f"train {len(train):,} rows | {train.is_laundering.sum():,} positives "      f"| {train.is_laundering.mean():.4%}")print(f"test  {len(test):,} rows | {test.is_laundering.sum():,} positives "      f"| {test.is_laundering.mean():.4%}")

In [ ]:
overview(train)

In [ ]:
target_balance(train)

**2,856 positives in 3.55M rows — 1 in 1,244.**At this imbalance ROC-AUC will look flattering and mean little. Headlinemetrics: **PR-AUC** and **recall at a fixed alert budget**.

---# 6. Raw columns

## 6.1 `payment_format`

In [ ]:
explore_categorical(train, "payment_format")

**Observed**| format | rows | positives | rate | lift ||---|---|---|---|---|| ACH | 400K | 2,381 | 0.60% | **7.3×** || Cheque | 1.24M | 226 | 0.02% | 0.2× || Credit Card | 883K | 139 | 0.02% | 0.2× || Cash | 328K | 74 | 0.02% | 0.3× || Bitcoin | 102K | 36 | 0.04% | 0.4× || Reinvestment | 481K | **0** | 0% | 0 || Wire | 116K | **0** | 0% | 0 |ACH holds **83% of all laundering** from 11% of rows.**Meaning.** The strongest categorical discriminator. The problem is closer to"which ACH transactions are suspicious" than "which transactions aresuspicious".**Decision.** Keep as a feature. Exclude Reinvestment and Wire from *trainingrows*, but keep all formats in the *scoring* path — production must scoreeverything, and zero-Wire-laundering is a simulator artifact. Wires are aprimary laundering channel in reality. Report recall per format at evaluation.

## 6.2 `amount_paid` — raw

In [ ]:
explore_numeric(train, "amount_paid")

In [ ]:
(train.groupby("currency_paid", observed=True)   .agg(n=("amount_paid","size"), median=("amount_paid","median"),        rate=("is_laundering","mean"), positives=("is_laundering","sum"))   .sort_values("n", ascending=False))

In [ ]:
usd = train[train.currency_paid == "US Dollar"]usd.groupby("is_laundering").amount_paid.describe([.25,.5,.75,.95]).T

**Observed.** Within US Dollar alone (holding currency constant):| percentile | normal | laundering | ratio ||---|---|---|---|| 25% | 153 | 1,898 | 12× || 50% | 960 | 4,995 | 5× || 75% | 6,024 | 12,817 | 2× || 95% | 224,277 | 27,523 | **0.12×** |**Meaning.** Not "laundering is larger" — laundering occupies a **band**,roughly 1.9K-28K. Rarely small *and* rarely very large. Normal traffic has botha mass of tiny amounts and a long tail into the millions; laundering hasneither.This is consistent with **structuring**: large enough to be worth moving, smallenough to avoid the scrutiny the largest transactions attract.Saudi Riyal shows a 3.7× rate on 61,910 rows — large enough to be real ratherthan noise, but most likely a simulator choice.**Decision.** Do not use raw amount as the primary signal; normalise withincurrency (§7.1). Retain raw `amount_paid` at no cost — trees may still finduseful splits.

## 6.3 Account attributes

In [ ]:
explore_categorical(train, "src_entity_type")explore_categorical(train, "src_bank_country")

**Observed.** Corporation 1.3×, Individual 1.6× (tiny volume), Saudi Arabia3.9×. Everything else ~1.0.**Meaning.** *Who* the account is barely matters. This finding is what pushedthe project toward behavioural and network features — the signal is in whataccounts **do**.**Decision.** Keep as weak context. Do not expect them to carry the model.

## 6.4 Time

In [ ]:
explore_temporal(train)

**Observed.**- Hour: rate peaks 11:00-16:00 (~0.0013) vs ~0.0002 at hour 0. **6× swing.**- Day of week: days 5-6 show the highest rates but the **lowest** volume.**Meaning.** Hour is a genuine pattern and knowable at scoring time —laundering blends into business hours where volume makes it less conspicuous.Day of week is not trustworthy: 10 days of data means 1-2 observations perweekday, and high rates coincide with low volumes. A property of which calendardays the simulation covered.**Decision.** Keep `hour_sin`/`hour_cos`/`is_hour_zero`. **Reject`day_of_week`.** Never use absolute date or days-since-file-start.

---# 7. Engineered features

## 7.1 `amt_pct_ccy` — amount percentile within currency

In [ ]:
explore_numeric(train, "amt_pct_ccy", log_scale=False)# redundancy check against the log-z alternativetrain["log_amount"] = np.log1p(train.amount_paid)gl = train.groupby("currency_paid", observed=True).log_amounttrain["amt_logz_ccy"] = (train.log_amount - gl.transform("mean")) / gl.transform("std")print(train[["amt_pct_ccy", "amt_logz_ccy"]].corr().round(3))

**Observed.** First clear class separation in the project. Normal transactionsare uniform by construction, so any departure from flat *is* the signal —laundering concentrates between 0.6 and 0.85.| decile | rate ||---|---|| 1 (smallest) | 0.00005 || 5 | 0.00085 || 9 | 0.0022 ← peak || 10 (largest) | 0.0003 ← **collapses** |**40× spread**, strongly **non-monotonic**.The two candidates correlate at **0.955**.**Decision.** Keep `amt_pct_ccy`, drop `amt_logz_ccy` and `log_amount`.Percentile rank is bounded 0-1, robust to outliers, and needs no fittedmean/std persisted alongside the model — one less train-serve skew risk.

## 7.2 `src_secs_since_last` — strongest feature

In [ ]:
explore_numeric(train, "src_secs_since_last", log_scale=True)

**Observed.**| gap | rate ||---|---|| < 2 min | 0.0015 || 5-15 min | 0.00002 || > 22 h | **0.0029** |**60× spread** — the widest of anything tested. Strongly **U-shaped**.**Meaning.** Two distinct behaviours at the two extremes:- **Very fast:** money arrives and leaves within minutes. The account is a  pass-through, not a destination — layering.- **Very slow:** a dormant account suddenly activating. Classic mule behaviour.- **The middle is normal:** someone paying bills over an afternoon.This feature correlates with nothing else above 0.40 — it carries genuinelyindependent information.

### Threshold selection — swept, not guessed

In [ ]:
base = train.is_laundering.mean()print("rapid:")for t in [10, 30, 60, 90, 300, 600]:    m = train.src_secs_since_last < t    print(f"  <{t:>4}s: n={m.sum():>8,} pos={train.is_laundering[m].sum():>5} "          f"lift={train.is_laundering[m].mean()/base:.2f}")print("dormant:")for t in [40_000, 80_000, 200_000, 300_000, 400_000, 500_000]:    m = train.src_secs_since_last > t    print(f"  >{t:>7,}s: n={m.sum():>8,} pos={train.is_laundering[m].sum():>5} "          f"lift={train.is_laundering[m].mean()/base:.2f}")

**Observed.**| rapid | lift | | dormant | positives | lift ||---|---|---|---|---|---|| ≤60s | 3.26 | | >200,000s | 570 | 10.33 || <90s | 2.24 | | >300,000s | 378 | 20.34 || <300s | 1.40 | | >400,000s | 208 | **24.60** || <600s | 0.97 | | >500,000s | 41 | 17.52 |Lift decays **smoothly** on both sides rather than jumping erratically —evidence of real structure rather than fitted noise. It turns over at 500,000swhere only 41 positives remain.**Timestamp resolution finding.** Thresholds of 10s, 30s and 60s return*byte-identical* results — timestamps are minute-granular. "≤60s" means *sameor adjacent minute*; sub-minute distinctions are not observable.**Decision.** `≤60s` and `>200,000s`. The dormant threshold trades peak lift(24.6× at 400,000s) for coverage, retaining 20% of all laundering ratherthan 7%.**Caveat.** 200,000s is ~23% of the 10-day observation window. These specificthresholds would need refitting on longer data, though the dormancy signalitself should generalise.

In [ ]:
train["src_rapid_txn"]    = train.src_secs_since_last <= 60train["src_dormant_wake"] = train.src_secs_since_last > 200_000explore_boolean(train, "src_rapid_txn")explore_boolean(train, "src_dormant_wake")

## 7.3 Network structure — fan-in and fan-out

In [ ]:
explore_numeric(train, "dst_fanin_ratio",  log_scale=False)explore_numeric(train, "src_fanout_ratio", log_scale=False)print("NaN share:", train.dst_fanin_ratio.isna().mean().round(3))

**What they measure.** Whether an account deals with many *different*counterparties or the same few repeatedly. A ratio near 1.0 means everycounterparty is new; near 0.1 means the same handful over and over.**Why it matters.** Laundering has a shape:- **fan-out (scatter):** one account splits money across many accounts —  breaking a large sum into inconspicuous pieces.- **fan-in (gather):** many accounts feed into one — reassembling it.A normal person pays the same landlord and utilities. A mule pays fiftyaccounts it has never dealt with.**Observed.**| feature | deciles 1→10 | spread | shape ||---|---|---|---|| `dst_fanin_ratio` | 0.00040 → **0.0035** | 8.7× | threshold (jumps above 0.667) || `src_fanout_ratio` | 0.00026 → 0.0011 | 4.2× | threshold (jumps above 0.486) |Neither is smoothly monotonic — both are flat until a sharp rise in the topone or two deciles.**Effect of the ≥3 guard.** Before it, laundering spiked at exactly 1.0, 0.5and 0.667 — those were 1/1, 1/2 and 2/3 from accounts with almost no history,reflecting newness rather than fan-out behaviour. The guard removed them.NaN share: 26%, which XGBoost handles natively.**Decision.** Keep both, plus an explicit `dst_high_fanin` flag since therelationship is a threshold rather than a gradient. These are the only featurescapturing **network shape** rather than properties of a single transaction —and network shape is what laundering fundamentally is.

In [ ]:
train["dst_high_fanin"] = train.dst_fanin_ratio > 0.667explore_boolean(train, "dst_high_fanin")

## 7.4 `src_log_dev` — deviation from the account's own baseline

In [ ]:
explore_numeric(train, "src_log_dev", log_scale=False)

**Observed.** Centred at 0 after the geometric-mean fix. Rate falls to 0.00037in decile 4 and rises to 0.0013 in decile 10 — **3.5× spread**, predominantlymonotonic.An absolute-value variant was also tested (treating unusually *small* the sameas unusually *large*) and scored only 2.2×. Direction carries most of thesignal: laundering amounts tend to be larger than the account's norm, notmerely different.**Decision.** Keep the signed version. Requires ≥3 prior transactions — a"typical amount" from one prior transaction is meaningless.

---# 8. Rejected featuresEvery rejection falls into one of four categories. None were dropped merely forbeing weak.

## 8.1 Artifact — `is_cross_currency` (hypothesis test)

Cross-currency movement is a recognised layering technique, so the priorexpectation was an **elevated** rate. The observed rate is zero — the oppositedirection, and worth testing formally.- **H₀:** laundering is independent of cross-currency status- **H₁:** the two are associated- **Test:** Fisher's exact (2×2; chosen over chi-square because the expected  count in the positive/cross-currency cell is small)- **α = 0.01**

In [ ]:
table = pd.crosstab(train.is_cross_currency, train.is_laundering)chi2, p_chi, dof, expected = chi2_contingency(table)odds, p_fisher = fisher_exact(table)print(table, "\n")print("expected under H0:\n", expected.round(1), "\n")print(f"chi-square : X2={chi2:.2f}, p={p_chi:.3e}")print(f"fisher     : OR={odds:.4f}, p={p_fisher:.3e}")

In [ ]:
explore_boolean(train, "is_cross_currency")explore_boolean(train, "is_self_loop")

**Result.** Under independence, 47,865 cross-currency rows would contain ~38.5positives. **Zero** were observed.- Chi-square: χ² = 38.01, p = 7.03e-10- Fisher's exact: OR = 0.0000, p = 3.47e-17H₀ rejected by both. The p-values differ by seven orders of magnitude — thechi-square approximation straining against an empty cell, which is preciselywhy Fisher was appropriate.**Interpretation.** The association is real, not sampling noise. Butsignificance establishes only that the pattern is unlikely under independence —it does not make the feature useful. Two reasons to reject it anyway:1. **Direction contradicts domain knowledge.** A protective effect from   currency conversion is not plausible behaviour.2. **A perfectly empty cell across 47,865 rows** is the signature of a   generator constraint. Real data would show *some* overlap.A model trained on this learns *"cross-currency ⇒ safe"* and wouldsystematically ignore exactly the transactions an investigator should review.**Decision.** Drop `is_cross_currency` and `amount_delta`. Exclude self-looprows (also 0 positives, 11.6% of data) from training.*The test quantifies the effect; domain judgement decides.***Note on multiple comparisons:** with many columns screened, at α = 0.05 onein twenty tests would appear significant by chance. α = 0.01 was usedthroughout for this reason.

## 8.2 Redundancy — correlation check

In [ ]:
CANDIDATES = [    "amt_pct_ccy", "src_secs_since_last_log",    "dst_fanin_ratio", "src_fanout_ratio", "src_log_dev",    "src_txn_count_prior", "dst_txn_count_prior",    "src_unique_dst_prior", "dst_unique_src_prior",    "hour",]corr = train[CANDIDATES].corr(method="spearman")fig, ax = plt.subplots(figsize=(9, 7))im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)ax.set_xticks(range(len(CANDIDATES))); ax.set_xticklabels(CANDIDATES, rotation=90)ax.set_yticks(range(len(CANDIDATES))); ax.set_yticklabels(CANDIDATES)for i in range(len(CANDIDATES)):    for j in range(len(CANDIDATES)):        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)plt.colorbar(im); plt.title("Spearman correlation")plt.tight_layout(); plt.show()high = (corr.abs()        .where(np.triu(np.ones(corr.shape), k=1).astype(bool))        .stack().sort_values(ascending=False))print("pairs above 0.7:"); print(high[high > 0.7])

Spearman rather than Pearson: these features are skewed and severalrelationships are monotonic but not linear.| pair | ρ | why ||---|---|---|| `src_unique_dst_prior` / `src_txn_count_prior` | 0.92 | numerator and denominator of the fan-out ratio || `dst_txn_count_prior` / `dst_fanin_ratio` | −0.81 | larger denominator drives the ratio down || `dst_txn_count_prior` / `dst_unique_src_prior` | 0.78 | same, incoming side || `amt_pct_ccy` / `src_log_dev` | 0.76 | both amount-based || `dst_fanin_ratio` / `src_fanout_ratio` | 0.72 | highly-connected accounts score high on both |**Decisions.**- **Drop the four raw counts.** They are the ratio components and add no  independent information.- **Keep both fan ratios** despite ρ = 0.72 — they describe opposite ends of a  chain. Expect their SHAP importances to split unpredictably.- **`src_log_dev` is marginal** (ρ = 0.76 with a far stronger feature).  Retained; test whether removing it changes PR-AUC.**On VIF:** not run. VIF detects multicollinearity that destabilises *linearmodel coefficients*. XGBoost has no coefficients and handles correlatedfeatures by picking one per split — prediction quality is unaffected. Pairwisecorrelation is sufficient for the redundancy check here.

## 8.3 Mechanical artifacts and superseded formulations

In [ ]:
for col in ["src_is_new", "dst_is_new"]:    if col not in train:        train[col] = train[f"{col[:3]}_txn_count_prior"] == 0    explore_boolean(train, col)

**`src_is_new` / `dst_is_new`.** Both show *lower* laundering (0.00033 and0.00020 vs 0.00088 base). Mechanical rather than behavioural — a firsttransaction has no history so cannot yet be part of a chain. The direction alsocontradicts real AML practice, where new accounts are elevated risk.**Rejected**, or kept only as a NaN indicator.**`src_amount_vs_own_mean`.** Max 3.5e7, U-shaped and unusable. Superseded by`src_log_dev`.**`src_log_dev_abs`.** 2.2× vs 3.5× signed — discards direction, which carriesmost of the signal.

---# 9. Final feature set

| feature | spread | shape | family ||---|---|---|---|| `src_secs_since_last_log` | **60×** | U-shaped | timing || `amt_pct_ccy` | **40×** | band | amount || `src_dormant_wake` | 10.3× | boolean | timing || `dst_fanin_ratio` | 8.7× | threshold | network || `payment_format` | 7.3× | categorical | raw || `hour_sin` / `hour_cos` | 6× | cyclical | time || `src_fanout_ratio` | 4.2× | threshold | network || `src_log_dev` | 3.5× | monotonic | behaviour || `src_rapid_txn` | 3.3× | boolean | timing || `dst_high_fanin` | — | boolean | network || `is_hour_zero` | — | boolean | data quality || entity type / bank country (×4) | ~1.3× | categorical | account |**Rejected, with reason**| feature | category | reason ||---|---|---|| `is_cross_currency`, `amount_delta` | artifact | 0 positives in 47,865 rows; χ² p=7e-10, Fisher p=3.5e-17. Real association, wrong direction || `is_self_loop` | artifact | 0 positives; exclude those rows instead || `amt_logz_ccy`, `log_amount` | redundant | ρ = 0.955 with `amt_pct_ccy` || four raw prior-counts | redundant | ρ up to 0.92 with the ratios || `src_log_dev_abs` | superseded | 2.2× vs 3.5× signed || `src_amount_vs_own_mean` | superseded | arithmetic mean inflated by outliers || `src_is_new`, `dst_is_new` | mechanical | wrong direction, no behavioural content || `day_of_week` | artifact | 10-day window, 1-2 observations per weekday |**Training row exclusions:** Reinvestment, Wire, self-loops, burn-in period.

In [ ]:
FEATURES = [    "amt_pct_ccy",    "src_secs_since_last_log", "src_rapid_txn", "src_dormant_wake",    "dst_fanin_ratio", "dst_high_fanin", "src_fanout_ratio",    "src_log_dev",    "hour_sin", "hour_cos", "is_hour_zero",    "payment_format",    "src_entity_type", "dst_entity_type",    "src_bank_country", "dst_bank_country",]print(len(FEATURES), "features")train[FEATURES].dtypes

---# 10. Implications for modelling**1. Not one top feature is linear-monotonic.**Amount is a band. Timing is U-shaped. Fan-in and fan-out are thresholds.Payment format is categorical. A linear model can only learn one direction —it cannot represent "both extremes are risky".→ **XGBoost as primary model. Logistic regression as an interpretablebaseline for comparison, not as a contender.****2. Metrics must suit a 0.08% positive rate.**→ **PR-AUC as headline, plus recall at a fixed alert budget** — ifinvestigators review 500 alerts a day, what share of laundering is caught?That is the question an AML team actually asks.**3. Cost asymmetry favours recall.**A missed STR is a regulatory failure; a false positive costs investigator time.Threshold selection should target a recall level and report the precision cost,rather than defaulting to 0.5.**4. A rules baseline comes first.**Detection rules are the incumbent in any real AML function. The model's resultshould be stated as lift *over rules*, not as a standalone AUC.

---# 11. Known limitations**Simulator artifacts.** Zero laundering on Wire, Reinvestment andcross-currency transactions are properties of the generator, not of banking.All three are genuine laundering channels in reality. Any model trained hereinherits those blind spots.**Window dependence.** 10 days of data. The dormancy threshold (200,000s ≈ 2.3days) is ~23% of the observation window and would need refitting on longerdata.**Timestamp resolution.** Minute-granular — sub-minute thresholds returnidentical results.**Chain truncation.** Chains beginning near day 10 are cut by the fileboundary, so some positives are partial patterns. The model trains largely oncompleted chains; in production it would see chains still in progress, andrecall on in-flight laundering would likely be lower.**Offline/online mismatch.** Features are computed batch-wise over fullhistory. Production would maintain account aggregates incrementally in afeature store. The definitions here are the specification the online store mustreproduce — keeping them consistent is the primary train-serve skew risk.---# 12. Next1. Port `FEATURES` into `features.py` with a leakage regression test.2. Build the rules baseline; measure its precision and recall.3. Train LR → RF → XGBoost; report PR-AUC and recall@k **against the   baseline**.4. Per-typology recall using the pattern labels — **evaluation only**. Those   labels are generator ground truth that would not exist at a real bank, so   they are never used for sampling, features, or model selection.